In [2]:
import torch
import torch.nn as nn

class VisualPerceptionModule(nn.Module):
    """
    EARLY FUSION STAGE:
    Combines RGB (Image) and Depth (Sensor) features immediately.
    """
    def __init__(self, rgb_dim, depth_dim):
        super(VisualPerceptionModule, self).__init__()
        # Concatenate dimensions: rgb_dim + depth_dim
        self.fusion_layer = nn.Sequential(
            nn.Linear(rgb_dim + depth_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256)
        )

    def forward(self, rgb, depth):
        # Concatenate at the input level (Early Fusion)
        x = torch.cat((rgb, depth), dim=1)
        return self.fusion_layer(x)

class HybridFusionModel(nn.Module):
    """
    HYBRID STAGE:
    Uses the output of Early Fusion and combines it with Audio via Late Fusion.
    """
    def __init__(self, rgb_dim=512, depth_dim=128, audio_dim=64, classes=3):
        super(HybridFusionModel, self).__init__()

        # Branch 1: The Early Fusion Module
        self.visual_branch = VisualPerceptionModule(rgb_dim, depth_dim)

        # Branch 2: Independent Audio Module
        self.audio_branch = nn.Sequential(
            nn.Linear(audio_dim, 128),
            nn.ReLU()
        )

        # LATE FUSION STAGE:
        # Merging the 'Visual Perception' and 'Audio' features
        self.classifier = nn.Sequential(
            nn.Linear(256 + 128, 128),
            nn.ReLU(),
            nn.Linear(128, classes)
        )

    def forward(self, rgb, depth, audio):
        # 1. Early Fusion of Vision and Depth
        visual_features = self.visual_branch(rgb, depth)

        # 2. Independent processing of Audio
        audio_features = self.audio_branch(audio)

        # 3. Late Fusion of the two branches
        combined = torch.cat((visual_features, audio_features), dim=1)

        return self.classifier(combined)


model = HybridFusionModel()
# Increased batch size for the sample inputs to resolve the BatchNorm1d error
sample_rgb = torch.randn(2, 512)
sample_depth = torch.randn(2, 128)
sample_audio = torch.randn(2, 64)

prediction = model(sample_rgb, sample_depth, sample_audio)
print(f"Hybrid Fusion Output Shape: {prediction.shape}")


Hybrid Fusion Output Shape: torch.Size([2, 3])
